# Experiment 2 — Structured Decomposition (Llama 3.1 8B via Ollama)

Run this experiment using the local **Ollama** runtime in Google Colab. It does not require Hugging Face gated-model access or a Hugging Face token. The model is downloaded from Ollama's model library. Meta's Llama licence still applies.

This notebook uses the full fixed 250-question obfuscated benchmark. All three experiment notebooks use the same 250 response IDs, question text, and ground-truth answers. This corrected version defaults to `TEST_MODE = False`, so the full 250 questions run. Set `TEST_MODE = True` only if you want a 5-question test first.

In [1]:
!nvidia-smi

Sat Oct 10 08:48:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip -q install "pandas==2.2.3" requests tqdm

In [3]:
import os
import re
import time
import random
import subprocess
import shutil
import requests
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# True = 5-question safety test (1 question per task/variant group)
# False = full 250-question inference (50 questions per task/variant group)
TEST_MODE = False
TEST_SAMPLES_PER_VARIANT = 1
SAMPLE_PER_VARIANT = 50

# Ollama downloads and runs a quantized Llama 3.1 8B model. No Hugging Face token is needed.
OLLAMA_MODEL = "llama3.1:8b"
MODEL_ID = "Ollama: llama3.1:8b (Q4_K_M)"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MAX_NEW_TOKENS = 500
MAX_RETRIES = 3
CHECKPOINT_EVERY = 1  # save after each attempted question

EXPERIMENT_NAME = "Structured Decomposition"
EXPERIMENT_SLUG = "structured_decomposition"
EXPERIMENT_NUMBER = "02"

print("Model:", MODEL_ID)
print("Experiment:", EXPERIMENT_NAME)
print("TEST_MODE:", TEST_MODE)
print("Ollama backend:", OLLAMA_BASE_URL)


Model: Ollama: llama3.1:8b (Q4_K_M)
Experiment: Structured Decomposition
TEST_MODE: False
Ollama backend: http://127.0.0.1:11434


In [4]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded.")
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise ValueError("Please upload the benchmark CSV.")

csv_name = csv_names[0]
df = pd.read_csv(csv_name)
print("Loaded:", csv_name)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head(3))

Saving qwen_obfuscated_benchmark_50_per_variant_250(1).csv to qwen_obfuscated_benchmark_50_per_variant_250(1).csv
Loaded: qwen_obfuscated_benchmark_50_per_variant_250(1).csv
Rows: 250
Columns: ['dataset', 'variant', 'setting', 'row_number', 'response_id', 'prompt', 'ground_truth']


,dataset,variant,setting,row_number,response_id,prompt,ground_truth
0,FOL,obfuscation,zero_shot,51,605365A270FD52AB,Respond with exactly one word without any addi...,FALSE
1,FOL,obfuscation,zero_shot,52,404832606BEEDC1E,Respond with exactly one word without any addi...,FALSE
2,FOL,obfuscation,zero_shot,53,35D50601BB7F7AF5,Respond with exactly one word without any addi...,TRUE


In [5]:
required_columns = [
    "dataset", "variant", "setting", "row_number", "response_id", "prompt", "ground_truth"
]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

expected = {
    ("FOL", "obfuscation"): 50,
    ("Blood_Relation", "obfuscation_l1"): 50,
    ("Blood_Relation", "obfuscation_l2"): 50,
    ("Direction_Sense", "obfuscation"): 50,
    ("Number_Series", "obfuscation"): 50,
}
if len(df) != 250:
    raise ValueError(f"Expected the fixed 250-row source benchmark, found {len(df)} rows.")
if df["response_id"].astype(str).duplicated().any():
    raise ValueError("response_id must be unique in the source benchmark.")

print("Source benchmark distribution:")
distribution = df.groupby(["dataset", "variant"]).size().reset_index(name="count")
display(distribution)
for key, expected_count in expected.items():
    actual = int(((df["dataset"].astype(str) == key[0]) & (df["variant"].astype(str) == key[1])).sum())
    print(f"{key[0]} / {key[1]}: {actual} / {expected_count}")
    if actual != expected_count:
        raise ValueError(f"Unexpected count for {key}: {actual}; expected {expected_count}.")

variant_order = [
    ("FOL", "obfuscation"),
    ("Blood_Relation", "obfuscation_l1"),
    ("Blood_Relation", "obfuscation_l2"),
    ("Direction_Sense", "obfuscation"),
    ("Number_Series", "obfuscation"),
]
selected_parts = []
for group_index, (dataset_name, variant_name) in enumerate(variant_order):
    group = df[
        (df["dataset"].astype(str) == dataset_name) &
        (df["variant"].astype(str) == variant_name)
    ].copy()
    # Include every row in each group (50 per group); no downsampling.
    picked = group.sort_values("row_number", kind="stable")
    selected_parts.append(picked)

shared_250 = pd.concat(selected_parts, ignore_index=True)
shared_250["benchmark_order"] = range(1, len(shared_250) + 1)
if len(shared_250) != 250 or shared_250["response_id"].astype(str).nunique() != 250:
    raise ValueError("Benchmark must contain exactly 250 unique response IDs.")

print("\nFull 250-question benchmark distribution:")
display(shared_250.groupby(["dataset", "variant"]).size().reset_index(name="count"))
display(shared_250[["benchmark_order", "dataset", "variant", "row_number", "response_id", "ground_truth"]])
shared_250.to_csv("llama_shared_250_question_benchmark.csv", index=False)
print("Saved sample file: llama_shared_250_question_benchmark.csv")

Source benchmark distribution:


,dataset,variant,count
0,Blood_Relation,obfuscation_l1,50
1,Blood_Relation,obfuscation_l2,50
2,Direction_Sense,obfuscation,50
3,FOL,obfuscation,50
4,Number_Series,obfuscation,50


FOL / obfuscation: 50 / 50
Blood_Relation / obfuscation_l1: 50 / 50
Blood_Relation / obfuscation_l2: 50 / 50
Direction_Sense / obfuscation: 50 / 50
Number_Series / obfuscation: 50 / 50

Full 250-question benchmark distribution:


,dataset,variant,count
0,Blood_Relation,obfuscation_l1,50
1,Blood_Relation,obfuscation_l2,50
2,Direction_Sense,obfuscation,50
3,FOL,obfuscation,50
4,Number_Series,obfuscation,50


,benchmark_order,dataset,variant,row_number,response_id,ground_truth
0,1,FOL,obfuscation,51,605365A270FD52AB,FALSE
1,2,FOL,obfuscation,52,404832606BEEDC1E,FALSE
2,3,FOL,obfuscation,53,35D50601BB7F7AF5,TRUE
3,4,FOL,obfuscation,54,58994DF26E347328,FALSE
4,5,FOL,obfuscation,55,17D02905EDAC2313,TRUE
...,...,...,...,...,...,...
245,246,Number_Series,obfuscation,96,10E562A00E5F7616,251.8
246,247,Number_Series,obfuscation,97,B66CBC94A1EBCD6C,441
247,248,Number_Series,obfuscation,98,05E7FEAC36C16B0E,24
248,249,Number_Series,obfuscation,99,980ED6E5BF8B0EA4,30


Saved sample file: llama_shared_250_question_benchmark.csv


In [6]:
STRUCTURED_DECOMPOSITION_SYSTEM = r'''
You are solving an obfuscated logical reasoning problem.

Use STRUCTURED DECOMPOSITION before solving.

1. IDENTIFY
   Identify every relevant entity, fact, relation, condition, quantity,
   movement, ordering, or sequence element in the problem.

2. STRUCTURE
   Convert the information into an explicit structured representation
   appropriate to the task.

   For relationship problems:
   - identify the people/entities
   - extract each relationship
   - connect the relationships into a chain or structure
   - preserve relationship direction

   For direction problems:
   - identify the starting point
   - list each movement in order
   - represent movements as directions/relative changes
   - combine them to determine the final position

   For number/sequence problems:
   - list the relevant sequence values
   - identify candidate transformations or patterns
   - verify the pattern across the available values

   For logical/FOL problems:
   - list the relevant entities, predicates, facts, and constraints
   - connect them according to the stated conditions

3. SOLVE
   Solve the question from the structured representation.
   Do not add facts that are not present.

4. VERIFY
   Check that the final answer is consistent with the original problem.

FINAL:
Return ONLY the final answer requested by the question.
Do not output the decomposition, reasoning, explanation, labels, or extra text.
'''

print("Structured Decomposition prompt ready.")

Structured Decomposition prompt ready.


In [7]:
!apt-get update -qq
!apt-get install -y zstd curl ca-certificates

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
curl is already the newest version (8.5.0-2ubuntu10.15).
ca-certificates is already the newest version (20260601~24.04.1).
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,116 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setti

In [8]:
# Install Ollama if the binary is not already available in this Colab runtime.
if shutil.which("ollama") is None:
    print("Installing Ollama...")
    install_proc = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        executable="/bin/bash",
        check=False,
    )
    if shutil.which("ollama") is None:
        raise RuntimeError(
            "Ollama installation did not produce an `ollama` executable. "
            "See the output above and retry in a fresh GPU runtime."
        )

# Start Ollama's local API server when it is not already running.
def ollama_is_ready():
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        return response.ok
    except requests.RequestException:
        return False

OLLAMA_SERVER_LOG = "/content/ollama_serve.log"
OLLAMA_SERVER_PROCESS = None
OLLAMA_SERVER_LOG_HANDLE = None

if not ollama_is_ready():
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    server_env["OLLAMA_NUM_PARALLEL"] = "1"
    server_env["OLLAMA_MAX_LOADED_MODELS"] = "1"
    OLLAMA_SERVER_LOG_HANDLE = open(OLLAMA_SERVER_LOG, "a")
    OLLAMA_SERVER_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=OLLAMA_SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        env=server_env,
        start_new_session=True,
    )
    for _ in range(60):
        if ollama_is_ready():
            break
        time.sleep(2)

if not ollama_is_ready():
    try:
        with open(OLLAMA_SERVER_LOG, "r") as log_file:
            print(log_file.read()[-6000:])
    except OSError:
        pass
    raise RuntimeError(
        "Ollama's local server did not become ready. Review the log above, "
        "then restart the Colab runtime and rerun this cell."
    )

print("Ollama API is ready.")
print(f"Pulling {OLLAMA_MODEL}; the quantized model download is about 4.9 GB. This may take several minutes...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)

# Confirm the model appears in Ollama's local model list.
tags_response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
tags_response.raise_for_status()
model_names = [item.get("name", "") for item in tags_response.json().get("models", [])]
if not any(name == OLLAMA_MODEL or name.startswith(OLLAMA_MODEL + ":") for name in model_names):
    raise RuntimeError(f"Model {OLLAMA_MODEL!r} not found after pull. Available models: {model_names}")
print("Model downloaded and available:", OLLAMA_MODEL)
print("No Hugging Face authentication is required for this Ollama download.")
print("Meta's Llama licence still applies to use of the model.")


Installing Ollama...
Ollama API is ready.
Pulling llama3.1:8b; the quantized model download is about 4.9 GB. This may take several minutes...
Model downloaded and available: llama3.1:8b
No Hugging Face authentication is required for this Ollama download.
Meta's Llama licence still applies to use of the model.


In [9]:
import unicodedata

def normalize_answer(value):
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()
    prefixes = [
        "Final Answer:",
        "Final answer:",
        "Answer:",
        "answer:",
        "The answer is:",
        "the answer is:",
    ]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
            break

    text = re.sub(r"\s+", " ", text).strip().lower()
    text = text.strip(" \t\n\r\"'.,;:!?")
    return text

def run_model(prompt, dataset=None):
    """Call the locally running Ollama chat API; no hosted API key is used."""
    if "SYSTEM_PROMPTS" in globals():
        # Experiment 3 provides task-specific system prompts via get_system_prompt().
        system_prompt = get_system_prompt(dataset)
    elif "NORMALIZE_SOLVE_SYSTEM" in globals():
        system_prompt = NORMALIZE_SOLVE_SYSTEM
    elif "STRUCTURED_DECOMPOSITION_SYSTEM" in globals():
        system_prompt = STRUCTURED_DECOMPOSITION_SYSTEM
    else:
        raise RuntimeError("No experiment system prompt was found.")

    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(prompt)},
        ],
        "stream": False,
        "keep_alive": "10m",
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_predict": MAX_NEW_TOKENS,
            "num_ctx": 8192,
        },
    }
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=(15, 900),
    )
    if not response.ok:
        raise RuntimeError(f"Ollama API error {response.status_code}: {response.text[:2000]}")
    data = response.json()
    answer = str(data.get("message", {}).get("content", "") or "").strip()
    if not answer:
        raise RuntimeError(f"Ollama returned an empty message. Response: {data}")
    return answer

print("Ollama inference and evaluation helpers ready.")


Ollama inference and evaluation helpers ready.


In [10]:
if TEST_MODE:
    # One test question per group = five questions total.
    parts = []
    for dataset_name, variant_name in variant_order:
        group = shared_250[
            (shared_250["dataset"].astype(str) == dataset_name) &
            (shared_250["variant"].astype(str) == variant_name)
        ]
        if len(group) < TEST_SAMPLES_PER_VARIANT:
            raise ValueError(f"Not enough test rows for {dataset_name}/{variant_name}.")
        parts.append(group.head(TEST_SAMPLES_PER_VARIANT))
    run_df = pd.concat(parts, ignore_index=True)
    run_label = "TEST"
    print("TEST MODE — 5 questions")
else:
    run_df = shared_250.copy().reset_index(drop=True)
    run_label = "250"
    print("FULL MODE — 250 questions")

print("Rows to process:", len(run_df))
display(run_df[["dataset", "variant", "row_number", "response_id", "ground_truth"]])

FULL MODE — 250 questions
Rows to process: 250


,dataset,variant,row_number,response_id,ground_truth
0,FOL,obfuscation,51,605365A270FD52AB,FALSE
1,FOL,obfuscation,52,404832606BEEDC1E,FALSE
2,FOL,obfuscation,53,35D50601BB7F7AF5,TRUE
3,FOL,obfuscation,54,58994DF26E347328,FALSE
4,FOL,obfuscation,55,17D02905EDAC2313,TRUE
...,...,...,...,...,...
245,Number_Series,obfuscation,96,10E562A00E5F7616,251.8
246,Number_Series,obfuscation,97,B66CBC94A1EBCD6C,441
247,Number_Series,obfuscation,98,05E7FEAC36C16B0E,24
248,Number_Series,obfuscation,99,980ED6E5BF8B0EA4,30


In [11]:
# Separate checkpoints for the safety test and full 250-question run.
checkpoint_file = f"experiment_02_structured_decomposition_Llama31_Ollama_{run_label}_checkpoint.csv"
results_file = f"experiment_02_structured_decomposition_Llama31_Ollama_{run_label}_results.csv"

if os.path.exists(checkpoint_file):
    old = pd.read_csv(checkpoint_file).fillna("")
    records_by_id = {str(row["response_id"]): row for row in old.to_dict("records")}
    print("Loaded checkpoint rows:", len(records_by_id))
else:
    records_by_id = {}
    print("No checkpoint found. Starting new run.")

for _, row in tqdm(run_df.iterrows(), total=len(run_df), desc="Structured Decomposition — Llama"):
    response_id = str(row["response_id"])
    previous = records_by_id.get(response_id, {})
    previous_prediction = str(previous.get("prediction", "") or "").strip()
    previous_status = str(previous.get("status", ""))
    if previous_prediction and previous_status == "success":
        continue  # keep already successful prediction

    raw_prediction = ""
    error = ""
    started = time.time()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if EXPERIMENT_NUMBER == "03":
                raw_prediction = run_model(row["prompt"], row["dataset"])
            else:
                raw_prediction = run_model(row["prompt"])
            if raw_prediction.strip():
                error = ""
                break
            error = "EMPTY_MODEL_RESPONSE"
        except Exception as exc:
            error = repr(exc)
            if attempt < MAX_RETRIES:
                wait_seconds = min(20, 2 ** attempt)
                print(f"\nError for {response_id} (attempt {attempt}/{MAX_RETRIES}): {exc}")
                print(f"Retrying in {wait_seconds} seconds...")
                time.sleep(wait_seconds)

    ground_truth_raw = str(row["ground_truth"])
    ground_truth_clean = normalize_answer(ground_truth_raw)
    prediction_clean = normalize_answer(raw_prediction)
    status = "success" if prediction_clean else "error"
    record = {
        "experiment": EXPERIMENT_NAME,
        "model": MODEL_ID,
        "dataset": str(row["dataset"]),
        "variant": str(row["variant"]),
        "setting": str(row.get("setting", "")),
        "row_number": str(row.get("row_number", "")),
        "response_id": response_id,
        "benchmark_order": int(row.get("benchmark_order", 0)),
        "ground_truth_raw": ground_truth_raw,
        "ground_truth": ground_truth_clean,
        "raw_prediction": raw_prediction,
        "prediction": prediction_clean,
        "correct": bool(ground_truth_clean and prediction_clean and prediction_clean == ground_truth_clean),
        "status": status,
        "error": error if status != "success" else "",
        "latency_seconds": round(time.time() - started, 3),
    }
    records_by_id[response_id] = record
    pd.DataFrame(list(records_by_id.values())).to_csv(checkpoint_file, index=False)

# Return rows in the same order as the intended sample, not checkpoint insertion order.
ordered_records = []
for _, row in run_df.iterrows():
    rid = str(row["response_id"])
    if rid in records_by_id:
        ordered_records.append(records_by_id[rid])
    else:
        gt_raw = str(row["ground_truth"])
        ordered_records.append({
            "experiment": EXPERIMENT_NAME, "model": MODEL_ID,
            "dataset": str(row["dataset"]), "variant": str(row["variant"]),
            "setting": str(row.get("setting", "")), "row_number": str(row.get("row_number", "")),
            "response_id": rid, "benchmark_order": int(row.get("benchmark_order", 0)),
            "ground_truth_raw": gt_raw, "ground_truth": normalize_answer(gt_raw),
            "raw_prediction": "", "prediction": "", "correct": False,
            "status": "error", "error": "NOT_RUN", "latency_seconds": 0.0,
        })

results_df = pd.DataFrame(ordered_records)
results_df.to_csv(results_file, index=False)
print("\nInference pass complete.")
print("Intended rows:", len(run_df))
print("Successful predictions:", int(results_df["prediction"].astype(str).str.strip().ne("").sum()))
print("Missing/failed predictions:", int(results_df["prediction"].astype(str).str.strip().eq("").sum()))
print("Checkpoint:", checkpoint_file)
print("Results:", results_file)

No checkpoint found. Starting new run.


Structured Decomposition — Llama:   0%|          | 0/250 [00:00<?, ?it/s]


Inference pass complete.
Intended rows: 250
Successful predictions: 250
Missing/failed predictions: 0
Checkpoint: experiment_02_structured_decomposition_Llama31_Ollama_250_checkpoint.csv
Results: experiment_02_structured_decomposition_Llama31_Ollama_250_results.csv


In [12]:
display(results_df[[
    "dataset", "variant", "row_number", "response_id",
    "ground_truth_raw", "raw_prediction", "ground_truth", "prediction", "correct", "status", "error"
]])

missing_count = int(results_df["prediction"].astype(str).str.strip().eq("").sum())
error_count = int(results_df["error"].astype(str).str.strip().ne("").sum())
print("Total intended rows:", len(run_df))
print("Predictions present:", int(results_df["prediction"].astype(str).str.strip().ne("").sum()))
print("Missing predictions:", missing_count)
print("Unresolved errors:", error_count)
if missing_count:
    print("⚠️ Rerun the inference cell after resolving the issue; successful rows will be skipped and missing rows retried.")
else:
    print("✅ All intended rows have a prediction.")

,dataset,variant,row_number,response_id,ground_truth_raw,raw_prediction,ground_truth,prediction,correct,status,error
0,FOL,obfuscation,51,605365A270FD52AB,FALSE,FALSE,false,false,True,success,
1,FOL,obfuscation,52,404832606BEEDC1E,FALSE,TRUE,false,true,False,success,
2,FOL,obfuscation,53,35D50601BB7F7AF5,TRUE,TRUE,true,true,True,success,
3,FOL,obfuscation,54,58994DF26E347328,FALSE,FALSE,false,false,True,success,
4,FOL,obfuscation,55,17D02905EDAC2313,TRUE,TRUE,true,true,True,success,
...,...,...,...,...,...,...,...,...,...,...,...
245,Number_Series,obfuscation,96,10E562A00E5F7616,251.8,5,251.8,5,False,success,
246,Number_Series,obfuscation,97,B66CBC94A1EBCD6C,441,"1. IDENTIFY\n - Relevant entities: integers,...",441,"1. identify - relevant entities: integers, md5...",False,success,
247,Number_Series,obfuscation,98,05E7FEAC36C16B0E,24,5,24,5,False,success,
248,Number_Series,obfuscation,99,980ED6E5BF8B0EA4,30,5,30,5,False,success,


Total intended rows: 250
Predictions present: 250
Missing predictions: 0
Unresolved errors: 0
✅ All intended rows have a prediction.


In [13]:
# Exact normalized match: useful as a first-pass score, but manually inspect equivalent-answer cases.
results_df["correct"] = results_df["correct"].astype(bool)
all_n = len(results_df)
successful = results_df[results_df["prediction"].astype(str).str.strip().ne("")].copy()
correct_n = int(results_df["correct"].sum())
accuracy_all = (correct_n / all_n * 100) if all_n else 0.0
accuracy_successful = (successful["correct"].mean() * 100) if len(successful) else 0.0
print(f"Exact-match correct: {correct_n}/{all_n}")
print(f"Accuracy over all requested rows: {accuracy_all:.2f}%")
print(f"Accuracy among successful predictions: {accuracy_successful:.2f}%" if len(successful) else "Accuracy among successful predictions: N/A")

summary = (
    results_df.groupby(["dataset", "variant"], dropna=False)
    .agg(
        requested=("response_id", "count"),
        predictions=("prediction", lambda s: int(s.astype(str).str.strip().ne("").sum())),
        correct=("correct", "sum"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )
    .reset_index()
)
summary["accuracy_over_requested_percent"] = (summary["correct"] / summary["requested"] * 100).round(2)
summary["accuracy_among_predictions_percent"] = summary.apply(
    lambda r: round(r["correct"] / r["predictions"] * 100, 2) if r["predictions"] else 0.0,
    axis=1,
)
display(summary)

Exact-match correct: 45/250
Accuracy over all requested rows: 18.00%
Accuracy among successful predictions: 18.00%


,dataset,variant,requested,predictions,correct,mean_latency_seconds,accuracy_over_requested_percent,accuracy_among_predictions_percent
0,Blood_Relation,obfuscation_l1,50,50,7,0.16022,14.0,14.0
1,Blood_Relation,obfuscation_l2,50,50,6,0.16228,12.0,12.0
2,Direction_Sense,obfuscation,50,50,1,0.77016,2.0,2.0
3,FOL,obfuscation,50,50,31,2.23776,62.0,62.0
4,Number_Series,obfuscation,50,50,0,3.46418,0.0,0.0


In [14]:
summary_file = f"experiment_02_structured_decomposition_Llama31_Ollama_{run_label}_summary.csv"
summary.to_csv(summary_file, index=False)
# Ensure the selected shared sample can be downloaded and checked against the other experiments.
shared_sample_file = "llama_ollama_shared_250_question_benchmark.csv"
shared_250.to_csv(shared_sample_file, index=False)
print("Saved results:", results_file)
print("Saved summary:", summary_file)
print("Saved shared 250-question benchmark:", shared_sample_file)

from google.colab import files
files.download(results_file)
time.sleep(1)
files.download(summary_file)
time.sleep(1)
files.download(shared_sample_file)

Saved results: experiment_02_structured_decomposition_Llama31_Ollama_250_results.csv
Saved summary: experiment_02_structured_decomposition_Llama31_Ollama_250_summary.csv
Saved shared 250-question benchmark: llama_ollama_shared_250_question_benchmark.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>